# NLP (Natural Language Processing)

NLP is used to process, represent, classify, search and predict from human language/text.

## Workflow

**Load → EDA → Text Cleaning → Train/Test Split → Text Representation → Train Model → Predict → Evaluate → Compare → Improve**

For NLP, preprocessing and text representation are the key steps.

Libraries / Tools

In [ ]:
# Common NLP libraries

# pip install nltk spacy scikit-learn
# pip install transformers

import pandas as pd
import numpy as np
import re

# Common tools:
# pandas        → data handling
# re            → text cleaning / regular expressions
# nltk          → tokenization, stopwords, stemming, etc.
# spaCy         → NLP preprocessing and pipelines
# scikit-learn  → TF-IDF + ML models
# transformers  → modern pretrained transformer models

Load Dataset

In [ ]:
df = pd.read_csv("text_data.csv")

df.head()

EDA

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# Check text length

df["text_length"] = df["text"].astype(str).str.len()

df["text_length"].describe()

## Text Preprocessing

Common steps:

- Convert to lowercase
- Remove HTML / unwanted symbols
- Remove extra whitespace
- Tokenization
- Stopword removal when appropriate
- Stemming or lemmatization when appropriate
- Handle missing text
- Keep preprocessing consistent between train and test

**Do not blindly remove everything.** In sentiment, negation words such as "not" can be important.

Clean Text

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

df[["text", "clean_text"]].head()

Handle Missing Values

In [ ]:
df["clean_text"] = df["clean_text"].fillna("")

print("Missing text:",
      df["clean_text"].isnull().sum())

Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

# Bag of Words

**Use:** Simple text representation where a document is represented by word counts.

Main idea:

**Text → vocabulary → word-count vector**

Limitation: word order/context is mostly ignored.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print("Train shape:", X_train_bow.shape)
print("Test shape:", X_test_bow.shape)

# TF-IDF

**Use:** Strong baseline for text classification/search because it gives more weight to informative words and less weight to very common words.

Main parameters:
- `max_features`
- `ngram_range`
- `min_df`
- `max_df`

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

# Logistic Regression

**Use:** Strong, simple baseline for text classification with sparse TF-IDF features.

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(
    max_iter=1000
)

model_lr.fit(X_train_tfidf, y_train)

pred_lr = model_lr.predict(X_test_tfidf)

print(pred_lr[:10])

# Naive Bayes

**Use:** Fast and effective baseline for many text-classification problems.

For count/TF-IDF text features, Multinomial Naive Bayes is a common choice.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model_nb = MultinomialNB()

model_nb.fit(
    X_train_tfidf,
    y_train
)

pred_nb = model_nb.predict(
    X_test_tfidf
)

print(pred_nb[:10])

# Linear SVM

**Use:** Often a strong classifier for high-dimensional sparse text features.

In [ ]:
from sklearn.svm import LinearSVC

model_svm = LinearSVC()

model_svm.fit(
    X_train_tfidf,
    y_train
)

pred_svm = model_svm.predict(
    X_test_tfidf
)

print(pred_svm[:10])

# KNN

**Use:** Possible for text, but usually less attractive for very high-dimensional sparse text data.

Useful mainly as a comparison/baseline when the dataset is suitable.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier(
    n_neighbors=5
)

model_knn.fit(
    X_train_tfidf,
    y_train
)

pred_knn = model_knn.predict(
    X_test_tfidf
)

print(pred_knn[:10])

# LDA Topic Modeling

**Use:** Unsupervised topic discovery, not ordinary text classification.

It tries to discover topics represented by groups of words.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42
)

X_topics = lda.fit_transform(
    X_train_bow
)

print("Document-topic shape:",
      X_topics.shape)

In [ ]:
# Display important words for each topic

feature_names = bow.get_feature_names_out()

for topic_id, topic in enumerate(lda.components_):
    top_words = topic.argsort()[-10:][::-1]
    words = [feature_names[i] for i in top_words]

    print(f"Topic {topic_id}:")
    print(words)

# Word Embeddings

**Use:** Represent words as dense numerical vectors that capture semantic relationships.

Common approaches:
- Word2Vec
- GloVe
- FastText

Unlike Bag of Words/TF-IDF, embeddings can represent semantic similarity more effectively.

In [ ]:
# Conceptual example

# Word2Vec / GloVe / FastText
#
# "king" → vector
# "queen" → vector
# "car" → vector
#
# Similar words tend to have related vector representations.

# Common library:
# gensim

# Transformer Models

**Use:** Modern NLP tasks where context and language understanding are important.

Examples:
- BERT
- RoBERTa
- DistilBERT
- T5
- GPT-style models

Transformers use attention to model relationships between tokens.

In [ ]:
# Example using Hugging Face Transformers

# pip install transformers torch

# from transformers import pipeline
#
# sentiment = pipeline("sentiment-analysis")
#
# result = sentiment("I really liked this product.")
#
# print(result)

## NLP Evaluation

For classification:
- **Accuracy**
- **Precision**
- **Recall**
- **F1**
- **Confusion Matrix**

For language-generation tasks, common metrics include:
- BLEU
- ROUGE

For topic modeling:
- Topic coherence
- Human/domain interpretation

For search/retrieval:
- Precision@K
- Recall@K
- MAP / NDCG

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_nlp(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": round(
            accuracy_score(y_true, y_pred), 3
        ),
        "Precision": round(
            precision_score(
                y_true, y_pred,
                average="weighted",
                zero_division=0
            ), 3
        ),
        "Recall": round(
            recall_score(
                y_true, y_pred,
                average="weighted",
                zero_division=0
            ), 3
        ),
        "F1": round(
            f1_score(
                y_true, y_pred,
                average="weighted",
                zero_division=0
            ), 3
        )
    }

In [ ]:
results = pd.DataFrame([
    evaluate_nlp("Logistic Regression", y_test, pred_lr),
    evaluate_nlp("Naive Bayes", y_test, pred_nb),
    evaluate_nlp("Linear SVM", y_test, pred_svm),
    evaluate_nlp("KNN", y_test, pred_knn)
])

results.sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

## Model Comparison

### Text Representation

**Bag of Words**
→ simple word counts

**TF-IDF**
→ weighted word importance

**Embeddings**
→ semantic dense representations

**Transformers**
→ contextual representations

### Classification Models

**Logistic Regression**
→ strong baseline

**Naive Bayes**
→ fast probabilistic text classifier

**Linear SVM**
→ strong for high-dimensional sparse text

**KNN**
→ simple similarity-based baseline

### Topic Modeling

**LDA**
→ discover topics without labels

# Model Improvement

**Baseline → Better Cleaning → Better Representation → Feature Engineering → Model Selection → Hyperparameter Tuning → Cross-Validation → Final Model**

### Text preprocessing
Try:
- Better tokenization
- Stopword strategy
- Lemmatization
- Handling negation
- Removing noisy text
- Normalizing text

### TF-IDF
Tune:
- `ngram_range`
- `max_features`
- `min_df`
- `max_df`
- `sublinear_tf`

### Classification
Tune:
- Logistic Regression `C`
- SVM `C`
- Naive Bayes smoothing parameters
- KNN `n_neighbors`

### Feature Engineering
Add:
- Word/character n-grams
- Text length
- Word count
- Punctuation features
- Domain-specific keywords

### Modern improvement
Move from:
**TF-IDF + classical ML**

to:
**Pretrained embeddings / Transformer models**

when the task and dataset justify the additional complexity.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

svm = LinearSVC()

params = {
    "C": [0.01, 0.1, 1, 10, 100]
}

search = RandomizedSearchCV(
    estimator=svm,
    param_distributions=params,
    n_iter=5,
    cv=5,
    scoring="f1_weighted",
    random_state=42
)

search.fit(
    X_train_tfidf,
    y_train
)

print("Best parameters:",
      search.best_params_)

print("Best CV F1:",
      search.best_score_)

In [ ]:
best_model = search.best_estimator_

final_pred = best_model.predict(
    X_test_tfidf
)

final_result = evaluate_nlp(
    "Tuned Linear SVM",
    y_test,
    final_pred
)

final_result

# Quick Revision

**Text Cleaning** → prepare raw text

**Tokenization** → split text into tokens

**Bag of Words** → word-count representation

**TF-IDF** → weighted word representation

**Word2Vec / GloVe / FastText** → dense word embeddings

**LDA** → topic modeling

**Logistic Regression** → strong TF-IDF baseline

**Naive Bayes** → fast text classification

**Linear SVM** → strong high-dimensional sparse-text classifier

**Transformers** → contextual language representation

### Main NLP workflow

**Raw Text → Clean → Split → Represent Text → Train → Predict → Evaluate → Compare → Tune → Final Model**